# Debug: Natural Gas Futures (`fetch_futures.py`)

Step-by-step debug notebook for the yfinance-based NG futures pipeline.

**Pipeline:** `pipeline/fetch_futures.py`  
**Output:** `data/futures.parquet`  
**Source:** Yahoo Finance (no API key required)

---
Run each cell in order. Each section validates one stage of the pipeline and prints a clear PASS/FAIL.

## 1. Imports & Dependencies

In [ ]:
import importlib
import sys
from datetime import date, timedelta
from pathlib import Path

# Check required packages
required = ["yfinance", "pandas", "pyarrow", "dateutil"]
missing = []
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print(f"FAIL  Missing packages: {missing}")
    print("      Run: pip install " + " ".join(missing))
else:
    import pandas as pd
    import yfinance as yf
    from dateutil.relativedelta import relativedelta
    print(f"PASS  All packages available")
    print(f"      pandas {pd.__version__}, yfinance {yf.__version__}")

## 2. Configuration

In [ ]:
ROLLING_YEARS = 5
NUM_DEFERRED = 13  # front month + 12 deferred
OUTPUT_PATH = Path("../data/futures.parquet")

MONTH_CODES = {
    1: "F", 2: "G", 3: "H", 4: "J", 5: "K", 6: "M",
    7: "N", 8: "Q", 9: "U", 10: "V", 11: "X", 12: "Z",
}

end_date = date.today()
start_date = end_date - timedelta(days=ROLLING_YEARS * 365)

print(f"Date range : {start_date} → {end_date}")
print(f"Output path: {OUTPUT_PATH.resolve()}")
print(f"Exists     : {OUTPUT_PATH.exists()}")

## 3. Ticker Construction

Verify the ticker-building logic produces correct CME codes for today's date.

In [ ]:
def build_ng_tickers(num_deferred: int = 13) -> list[str]:
    tickers = ["NG=F"]
    today = date.today()
    for offset in range(num_deferred):
        target = today + relativedelta(months=offset)
        code = MONTH_CODES[target.month]
        year_suffix = str(target.year)[-2:]
        tickers.append(f"NG{code}{year_suffix}.NYM")
    return tickers

tickers = build_ng_tickers(NUM_DEFERRED)
print(f"PASS  Built {len(tickers)} tickers")
print()
for t in tickers:
    print(f"  {t}")

## 4. Connectivity Test — Continuous Contract (NG=F)

Fetch 5 days of the continuous contract to verify Yahoo Finance is reachable.

In [ ]:
test_start = (date.today() - timedelta(days=7)).isoformat()
test_end = date.today().isoformat()

print(f"Fetching NG=F from {test_start} to {test_end} ...")
try:
    raw = yf.download("NG=F", start=test_start, end=test_end,
                      auto_adjust=True, progress=False)
    if raw.empty:
        print("FAIL  No data returned — Yahoo Finance may be rate-limiting or market is closed")
    else:
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.droplevel(1)
        print(f"PASS  {len(raw)} trading days returned")
        print()
        print(raw[["Open", "High", "Low", "Close", "Volume"]])
except Exception as e:
    print(f"FAIL  Exception: {e}")

## 5. Fetch All Tickers

Run the full ticker sweep. Expired or unlisted contracts will be skipped silently.

In [ ]:
def fetch_ohlcv(ticker: str, start: str, end: str) -> pd.DataFrame:
    raw = yf.download(ticker, start=start, end=end,
                      auto_adjust=True, progress=False)
    if raw.empty:
        return pd.DataFrame()
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.index.name = "Date"
    df["ticker"] = ticker
    return df

start_str = start_date.isoformat()
end_str = end_date.isoformat()

frames = []
skipped = []

for ticker in tickers:
    df = fetch_ohlcv(ticker, start_str, end_str)
    if df.empty:
        skipped.append(ticker)
        print(f"  SKIP  {ticker}")
    else:
        frames.append(df)
        print(f"  OK    {ticker:20s}  {len(df):4d} rows  "
              f"{df.index.min().date()} → {df.index.max().date()}")

print()
print(f"Fetched : {len(frames)} tickers")
print(f"Skipped : {len(skipped)} tickers ({', '.join(skipped) or 'none'})")

## 6. Combine & Inspect

In [ ]:
if not frames:
    print("FAIL  No data frames to combine — check connectivity and ticker list")
else:
    combined = pd.concat(frames).sort_index()
    print(f"PASS  Combined DataFrame: {combined.shape[0]} rows × {combined.shape[1]} cols")
    print()
    print("Schema:")
    print(combined.dtypes)
    print()
    print("Tickers present:")
    print(combined["ticker"].value_counts().to_string())
    print()
    print("Date range per ticker:")
    print(combined.groupby("ticker")["Close"].agg(["count", "first", "last"]))

## 7. Data Quality Checks

In [ ]:
if frames:
    issues = []

    # Check for NaN close prices
    nan_close = combined["Close"].isna().sum()
    if nan_close > 0:
        issues.append(f"NaN Close prices: {nan_close} rows")

    # Check for negative prices
    neg_prices = (combined[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum()
    if neg_prices > 0:
        issues.append(f"Non-positive prices: {neg_prices} rows")

    # Check continuous contract covers full window
    ng_f = combined[combined["ticker"] == "NG=F"]
    if ng_f.empty:
        issues.append("NG=F continuous contract has NO data")
    else:
        coverage_days = (ng_f.index.max() - ng_f.index.min()).days
        expected_days = ROLLING_YEARS * 365
        if coverage_days < expected_days * 0.9:
            issues.append(f"NG=F only covers {coverage_days} days (expected ~{expected_days})")

    # Check for duplicate rows
    dupes = combined.duplicated(subset=["ticker"]).sum()
    # (same date + ticker combo)
    dupes2 = combined.reset_index().duplicated(subset=["Date", "ticker"]).sum()
    if dupes2 > 0:
        issues.append(f"Duplicate (Date, ticker) rows: {dupes2}")

    if issues:
        print("WARN  Issues found:")
        for i in issues:
            print(f"      - {i}")
    else:
        print("PASS  No data quality issues detected")

    print()
    print("Sample (last 5 rows of NG=F):")
    print(combined[combined["ticker"] == "NG=F"].tail(5).to_string())

## 8. Compare to Saved Parquet (if it exists)

In [ ]:
if OUTPUT_PATH.exists():
    saved = pd.read_parquet(OUTPUT_PATH)
    print(f"Saved parquet: {saved.shape[0]} rows, last date = {saved.index.max()}")
    print()
    print("Saved tickers:")
    print(saved["ticker"].value_counts().to_string())
    print()
    if frames:
        fresh_max = combined.index.max()
        saved_max = saved.index.max()
        lag = (fresh_max - saved_max).days
        if lag > 5:
            print(f"WARN  Saved parquet is {lag} days behind fresh fetch")
        else:
            print(f"PASS  Saved parquet is up to date (lag = {lag} days)")
else:
    print(f"INFO  No saved parquet at {OUTPUT_PATH} — run pipeline to generate it")

## 9. Write Output (optional)

Uncomment to write fresh data to the parquet file.

In [ ]:
# if frames:
#     OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     combined.to_parquet(OUTPUT_PATH, index=True)
#     print(f"Wrote {len(combined)} rows → {OUTPUT_PATH}")